# Penyelesaian Sistem Persamaan Linear Tiga Variabel dengan Metode Eliminasi Gauss

**Mata kuliah:** Komputasi Numerik  
**Nama:** Esterlina Rahab Dos Santos  
**NIM:** 202511420081  

Notebook ini menyelesaikan sistem persamaan linear (SPL) tiga variabel menggunakan algoritma Eliminasi Gauss dan substitusi mundur. Setiap bagian dipisahkan ke dalam cell agar dapat dijalankan secara berurutan di Google Colab.

## Cell 1 — Tujuan dan model matematika

Contoh SPL yang digunakan:

\[
\begin{aligned}
2x+y-z &= 8 \\
-3x-y+2z &= -11 \\
-2x+y+2z &= -3
\end{aligned}
\]

Bentuk matriksnya adalah \(A\mathbf{x}=\mathbf{b}\), dengan matriks koefisien \(A\), vektor variabel \(\mathbf{x}\), dan vektor konstanta \(\mathbf{b}\).

**Penjelasan:** Kita mengubah persamaan menjadi matriks augmented [A|b]. Eliminasi Gauss menghilangkan elemen di bawah diagonal utama hingga terbentuk matriks segitiga atas. Setelah itu, substitusi mundur digunakan untuk memperoleh nilai variabel.

## Cell 2 — Menyiapkan data

Matriks `A` berisi koefisien x, y, dan z. List `b` berisi konstanta di ruas kanan. Data contoh ini dapat diganti dengan koefisien SPL lain yang memiliki solusi tunggal.

In [1]:
# Matriks koefisien dari x, y, dan z
A = [
    [2.0,  1.0, -1.0],
    [-3.0, -1.0, 2.0],
    [-2.0,  1.0, 2.0]
]

# Vektor konstanta pada ruas kanan persamaan
b = [8.0, -11.0, -3.0]

# Menampilkan data awal agar input dapat diperiksa
print("Matriks koefisien A:")
for baris in A:
    print(baris)

print("\\nVektor konstanta b:")
print(b)

Matriks koefisien A:
[2.0, 1.0, -1.0]
[-3.0, -1.0, 2.0]
[-2.0, 1.0, 2.0]
\nVektor konstanta b:
[8.0, -11.0, -3.0]


## Cell 3 — Fungsi Eliminasi Gauss

Fungsi berikut membuat salinan data agar data asli tidak berubah. Pada setiap kolom, program memilih pivot dengan nilai absolut terbesar dari baris yang tersedia (partial pivoting), menukar baris bila diperlukan, lalu menghilangkan elemen-elemen di bawah pivot.

**Keterangan penting:**
- `n` adalah jumlah persamaan/variabel.
- `pivot_row` adalah baris pivot saat ini.
- `factor` adalah pengali untuk membuat elemen di bawah pivot menjadi nol.
- Pemeriksaan pivot nol membantu mendeteksi sistem yang tidak memiliki solusi tunggal.

In [2]:
def eliminasi_gauss(A, b):
    # Menyalin matriks dan konstanta agar input asli tetap aman
    M = [baris[:] for baris in A]
    rhs = b[:]
    n = len(M)

    # Memastikan ukuran sistem sesuai untuk SPL persegi
    if n == 0 or any(len(baris) != n for baris in M) or len(rhs) != n:
        raise ValueError("A harus matriks persegi dan panjang b harus sama dengan jumlah baris A.")

    # Memproses setiap kolom pivot, kecuali kolom terakhir
    for kolom in range(n - 1):
        # Memilih baris dengan nilai pivot absolut terbesar
        pivot_row = max(range(kolom, n), key=lambda r: abs(M[r][kolom]))

        # Menolak pivot yang terlalu dekat dengan nol
        if abs(M[pivot_row][kolom]) < 1e-12:
            raise ValueError("Sistem tidak memiliki solusi tunggal (pivot nol).")

        # Menukar baris matriks dan konstanta jika pivot bukan di baris aktif
        if pivot_row != kolom:
            M[kolom], M[pivot_row] = M[pivot_row], M[kolom]
            rhs[kolom], rhs[pivot_row] = rhs[pivot_row], rhs[kolom]

        # Menghilangkan nilai di bawah pivot
        for baris in range(kolom + 1, n):
            factor = M[baris][kolom] / M[kolom][kolom]
            M[baris][kolom] = 0.0

            # Memperbarui elemen di sisi kanan pivot
            for j in range(kolom + 1, n):
                M[baris][j] -= factor * M[kolom][j]

            # Memperbarui konstanta pada baris yang sama
            rhs[baris] -= factor * rhs[kolom]

    # Memeriksa pivot terakhir sebelum mengembalikan hasil
    if abs(M[n - 1][n - 1]) < 1e-12:
        raise ValueError("Sistem tidak memiliki solusi tunggal (pivot terakhir nol).")

    return M, rhs


# Menjalankan eliminasi dan menampilkan matriks segitiga atas
U, c = eliminasi_gauss(A, b)

print("Matriks segitiga atas:")
for i in range(len(U)):
    print([round(nilai, 6) for nilai in U[i]], "|", round(c[i], 6))

Matriks segitiga atas:
[-3.0, -1.0, 2.0] | -11.0
[0.0, 1.666667, 0.666667] | 4.333333
[0.0, 0.0, 0.2] | -0.2


## Cell 4 — Substitusi mundur

Setelah eliminasi, sistem berbentuk segitiga atas. Nilai variabel dihitung dari baris paling bawah menuju baris paling atas.

Rumus umum:

\[
x_i=\frac{b_i-\sum_{j=i+1}^{n-1}a_{ij}x_j}{a_{ii}}
\]

Program menyimpan hasil setiap variabel di dalam list `x`.

In [10]:
def substitusi_mundur(U, c):
    # Menentukan jumlah variabel
    n = len(U)

    # Menyiapkan tempat untuk menyimpan nilai variabel
    x = [0.0] * n

    # Menghitung variabel dari baris terakhir ke baris pertama
    for i in range(n - 1, -1, -1):
        jumlah = 0.0

        # Menjumlahkan kontribusi variabel yang sudah diketahui
        for j in range(i + 1, n):
            jumlah += U[i][j] * x[j]

        # Memastikan elemen diagonal tidak nol
        if abs(U[i][i]) < 1e-12:
            raise ValueError("Tidak dapat melakukan substitusi: elemen diagonal nol.")

        # Menghitung nilai variabel pada baris ke-i
        x[i] = (c[i] - jumlah) / U[i][i]

    return x


# Menghitung solusi SPL
solusi = substitusi_mundur(U, c)

## Cell 5 — Menampilkan hasil

Hasil solusi ditampilkan dengan label x, y, dan z. Format enam angka di belakang koma digunakan agar hasil mudah dibaca.

In [6]:
nama_variabel = ["x", "y", "z"]

print("Hasil penyelesaian SPL:")
for nama, nilai in zip(nama_variabel, solusi):
    print(f"{nama} = {nilai:.6f}")

Hasil penyelesaian SPL:
x = 2.000000
y = 3.000000
z = -1.000000


## Cell 6 — Verifikasi hasil

Verifikasi dilakukan dengan memasukkan kembali nilai solusi ke persamaan awal. Program menghitung ruas kiri setiap persamaan dan membandingkannya dengan konstanta pada ruas kanan.

Jika selisihnya mendekati nol, solusi konsisten dengan sistem awal.

In [7]:
print("Verifikasi ke persamaan awal:")
for i in range(len(A)):
    ruas_kiri = sum(A[i][j] * solusi[j] for j in range(len(solusi)))
    selisih = ruas_kiri - b[i]
    print(
        f"Persamaan {i + 1}: "
        f"ruas kiri = {ruas_kiri:.6f}, "
        f"ruas kanan = {b[i]:.6f}, "
        f"selisih = {selisih:.2e}"
    )

Verifikasi ke persamaan awal:
Persamaan 1: ruas kiri = 8.000000, ruas kanan = 8.000000, selisih = 0.00e+00
Persamaan 2: ruas kiri = -11.000000, ruas kanan = -11.000000, selisih = 0.00e+00
Persamaan 3: ruas kiri = -3.000000, ruas kanan = -3.000000, selisih = 8.88e-16


## Cell 7 — Kesimpulan

Untuk contoh SPL pada notebook ini, metode Eliminasi Gauss menghasilkan nilai variabel yang memenuhi ketiga persamaan. Tahap eliminasi membentuk matriks segitiga atas, sedangkan substitusi mundur menentukan nilai variabel dari bawah ke atas.

**Catatan:** Program ini dirancang untuk SPL persegi dengan solusi tunggal. Jika pivot nol atau sangat dekat dengan nol, program memberikan pesan kesalahan daripada menampilkan hasil yang tidak dapat dipertanggungjawabkan.